In [ ]:
import torch
torch.__version__

In [ ]:
!rm -rf kernelforge/
!git clone https://github.com/marcalph/kernelforge.git

In [ ]:
# wurlitzer -> capture c level stdout/err, ninja -> cpp build tool
!pip install wurlitzer ninja torchvision
%load_ext wurlitzer

# Grayscale

### python version

In [ ]:
import torchvision.transforms.functional as tvf
from torchvision import io
from kernelforge.src.py.grayscale import show_img, rgb2gray_py
img = io.read_image('./kernelforge/pisco.jpg') # CHW
img_small = tvf.resize(img, 150, antialias=True)
ch,h,w = img_small.shape
show_img(img_small)



In [ ]:
%%time
img_g = rgb2gray_py(img_small)
     


In [ ]:
img_g = rgb2gray_py(img_small)
print(img_g.shape)
show_img(img_g, cmap='gray')
     


### CUDA
!rm -rf /root/.cache/torch_extensions/py312_cu128/

In [ ]:
!rm -rf /root/.cache/torch_extensions/py312_cu128/

In [ ]:
!ls

In [ ]:
from pathlib import Path
from kernelforge.src.py.compile import compile_extension
from kernelforge.src.py.grayscale import main

module = compile_extension("kernelforge/src/kernels/grayscale.cu")
res = main(module)


In [ ]:
imgc = img.permute(1,2,0).contiguous().cuda()
imgc.shape


In [ ]:
%time
res = module.grayscale(imgc).cpu()
res.shape

In [ ]:
show_img(res.permute(2,0,1), cmap='gray')

# Blur

In [ ]:
from kernelforge.src.py.blur import main
from kernelforge.src.py.compile import compile_extension

module = compile_extension(
    cuda_source = "kernelforge/src/kernels/blur.cu",
    cpp_source = "torch::Tensor blur(torch::Tensor image, int radius);"
    )
blurred = main(module)


In [ ]:
%time
res = module.blur(imgc, 8).cpu()
res.shape

In [ ]:

show_img(blurred.permute(2,0,1))